# 6. Drifting vs Diffusion, Flow Matching, GANs and MMD

Everything so far has been internal to drifting. This notebook puts it next to the four paradigms it is closest to,
on **one target, one generator architecture, one compute budget**, and asks three questions:

1. What is each method's training signal, mechanically?
2. What does each cost at inference?
3. Where does each break?

The interesting comparisons are not the headline numbers — they are two structural results we will derive and then
verify numerically:

* **The gradient of MMD *is* the drifting field, minus the softmax normalisation** (§3). Drifting is not "MMD with
  tricks"; it is the same interaction, made scale-free. That single change is what fixes MMD's vanishing-gradient
  pathology.
* **Drifting is a GAN whose discriminator has been replaced by a fixed kernel** (§4). Repulsion does the job the
  discriminator's gradient does — push samples apart, off each other — without the minimax game, so there is no
  training instability and no mode collapse to speak of.

Diffusion and flow matching sit in a different family entirely: they buy stability with a *time variable* and pay for
it with $N$ network evaluations at inference (§5).

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. One target, one generator, five training signals

The target is **eight Gaussians on a ring**. It is the standard mode-collapse probe: modes are well separated, so a
model that drops one is obvious, and "coverage" is a countable quantity rather than a vibe.

In [ ]:
N_MODES, RADIUS, SIGMA = 8, 1.0, 0.075
ANG = torch.arange(N_MODES) * 2 * math.pi / N_MODES
CENTRES = torch.stack([RADIUS * ANG.cos(), RADIUS * ANG.sin()], -1).to(device)

def ring(n, device=device):
    idx = torch.randint(0, N_MODES, (n,), device=device)
    return CENTRES[idx] + SIGMA * torch.randn(n, 2, device=device)

# ---- shared architecture: the same MLP body everywhere -----------------------
def mlp(d_in, d_out, h=256, nl=3, act=nn.SiLU):
    L, d = [], d_in
    for _ in range(nl):
        L += [nn.Linear(d, h), act()]; d = h
    return nn.Sequential(*L, nn.Linear(d, d_out))

class Gen(nn.Module):            # z -> x   (drifting, GAN, MMD)
    def __init__(self, d_z=2, d_out=2): super().__init__(); self.net = mlp(d_z, d_out)
    def forward(self, z): return self.net(z)

class TimeNet(nn.Module):        # (x, t) -> vector   (DDPM eps, flow velocity)
    def __init__(self, d=2, h=256):
        super().__init__()
        self.emb = nn.Sequential(nn.Linear(1, h), nn.SiLU(), nn.Linear(h, h))
        self.net = mlp(d + h, d, h=h)
    def forward(self, x, t):
        return self.net(torch.cat([x, self.emb(t.view(-1, 1))], -1))

# ---- metrics ----------------------------------------------------------------
def tv2d(A, B, bins=40, lim=1.5):
    def h(z):
        H = torch.histogramdd(z.cpu(), bins=[bins, bins], range=[-lim, lim, -lim, lim])[0]
        return H / H.sum()
    return 0.5 * (h(A) - h(B)).abs().sum().item()

def mode_stats(x):
    """How many of the 8 modes are populated, and how uneven is the split?"""
    d = torch.cdist(x, CENTRES)
    near = d.min(1)
    assign = near.indices[near.values < 4 * SIGMA]              # ignore off-manifold points
    counts = torch.bincount(assign, minlength=N_MODES).float()
    p = counts / counts.sum().clamp_min(1)
    covered = int((p > 0.01).sum())                            # a mode with <1% mass is "dropped"
    ent = -(p.clamp_min(1e-12) * p.clamp_min(1e-12).log()).sum() / math.log(N_MODES)
    onman = (near.values < 4 * SIGMA).float().mean().item()
    return covered, ent.item(), onman

ref = ring(20000)
print("reference: covered=%d  balance=%.3f  on-manifold=%.3f  TV(self)=%.3f"
      % (*mode_stats(ref), tv2d(ring(20000), ref)))

Three numbers per method, so that failure modes are distinguishable:

* **TV** — total-variation distance on a $40\times40$ histogram. Overall distribution error. The self-distance above
  (two independent draws of 20k points) is the noise floor; nothing can beat it.
* **covered** — how many of the 8 modes carry $>1\%$ of the mass. This is the mode-collapse detector.
* **balance** — normalised entropy of the mode histogram, 1.0 = perfectly even. Catches *partial* collapse, where a
  mode is technically present but starved.

We also track **on-manifold**: the fraction of samples within $4\sigma$ of any mode. A model can cover every mode and
still smear mass through the empty middle of the ring; that shows up here and not in the other three.

### 1.1 DDPM

$N$ steps at inference, denoising regression at training. The reference point.

In [ ]:
T = 200
betas = torch.linspace(1e-4, 0.02, T, device=device)
abar = torch.cumprod(1 - betas, 0)

def train_ddpm(steps=4000, N=256, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    net = TimeNet().to(device)
    opt = torch.optim.Adam(net.parameters(), lr)
    for _ in range(steps):
        x0 = ring(N)
        t = torch.randint(0, T, (N,), device=device)
        eps = torch.randn_like(x0)
        xt = abar[t].sqrt()[:, None] * x0 + (1 - abar[t]).sqrt()[:, None] * eps
        loss = F.mse_loss(net(xt, t.float() / T), eps)
        opt.zero_grad(); loss.backward(); opt.step()
    return net

@torch.no_grad()
def sample_ddpm(net, n, n_steps=T):
    """Ancestral sampling on a strided subsequence of the T-step chain (DDIM-style eta=1)."""
    x = torch.randn(n, 2, device=device)
    idx = torch.linspace(T - 1, 0, n_steps).long().to(device)
    for i, t in enumerate(idx):
        eps = net(x, t.float().expand(n) / T)
        a_t = abar[t]
        a_prev = abar[idx[i + 1]] if i + 1 < len(idx) else torch.tensor(1.0, device=device)
        x0 = (x - (1 - a_t).sqrt() * eps) / a_t.sqrt()
        sig = ((1 - a_prev) / (1 - a_t) * (1 - a_t / a_prev)).clamp_min(0).sqrt()
        x = a_prev.sqrt() * x0 + (1 - a_prev - sig**2).clamp_min(0).sqrt() * eps
        if i + 1 < len(idx):
            x = x + sig * torch.randn_like(x)
    return x

### 1.2 Flow matching

Same time variable, deterministic transport: regress the velocity of the straight-line interpolant
$x_t=(1-t)\epsilon + t\,x_0$, then solve the ODE.

In [ ]:
def train_flow(steps=4000, N=256, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    net = TimeNet().to(device)
    opt = torch.optim.Adam(net.parameters(), lr)
    for _ in range(steps):
        x0, z = ring(N), torch.randn(N, 2, device=device)
        t = torch.rand(N, 1, device=device)
        xt = (1 - t) * z + t * x0
        loss = F.mse_loss(net(xt, t), x0 - z)          # target velocity is constant along the path
        opt.zero_grad(); loss.backward(); opt.step()
    return net

@torch.no_grad()
def sample_flow(net, n, n_steps=50):
    x = torch.randn(n, 2, device=device)
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.full((n, 1), i * dt, device=device)
        x = x + dt * net(x, t)
    return x

### 1.3 GAN

1 NFE, but the training signal comes from a network that is itself being trained against you. Non-saturating loss,
$R_1$ gradient penalty (without it this collapses on a 2-D ring within a few hundred steps).

In [ ]:
def train_gan(steps=4000, N=256, lr=2e-4, r1=0.5, seed=0):
    torch.manual_seed(seed)
    g, d = Gen().to(device), mlp(2, 1).to(device)
    og = torch.optim.Adam(g.parameters(), lr, betas=(0.5, 0.9))
    od = torch.optim.Adam(d.parameters(), lr, betas=(0.5, 0.9))
    for _ in range(steps):
        # --- discriminator
        real = ring(N).requires_grad_(True)
        fake = g(torch.randn(N, 2, device=device)).detach()
        dr, df = d(real), d(fake)
        gp = torch.autograd.grad(dr.sum(), real, create_graph=True)[0].pow(2).sum(-1).mean()
        ld = F.softplus(-dr).mean() + F.softplus(df).mean() + 0.5 * r1 * gp
        od.zero_grad(); ld.backward(); od.step()
        # --- generator (non-saturating)
        fake = g(torch.randn(N, 2, device=device))
        lg = F.softplus(-d(fake)).mean()
        og.zero_grad(); lg.backward(); og.step()
    return g

### 1.4 MMD (generative moment matching network)

1 NFE, non-adversarial, and — this is the point of §3 — the closest relative drifting has. Unbiased MMD$^2$ with a
mixture of RBF bandwidths, minimised directly.

In [ ]:
BW = (0.05, 0.1, 0.25, 0.5, 1.0)      # bandwidth mixture, sigma (not sigma^2)

def mmd2(x, y, bws=BW):
    def K(a, b):
        d2 = torch.cdist(a, b).pow(2)
        return sum(torch.exp(-d2 / (2 * s * s)) for s in bws)
    n, m = x.shape[0], y.shape[0]
    kxx, kyy, kxy = K(x, x), K(y, y), K(x, y)
    kxx = kxx - torch.diag_embed(kxx.diagonal())      # unbiased: drop the diagonal
    kyy = kyy - torch.diag_embed(kyy.diagonal())
    return kxx.sum() / (n * (n - 1)) + kyy.sum() / (m * (m - 1)) - 2 * kxy.mean()

def train_mmd(steps=4000, N=256, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    g = Gen().to(device)
    opt = torch.optim.Adam(g.parameters(), lr)
    for _ in range(steps):
        loss = mmd2(g(torch.randn(N, 2, device=device)), ring(N))
        opt.zero_grad(); loss.backward(); opt.step()
    return g

### 1.5 Drifting

The Notebook 3 recipe, unchanged: multi-temperature, feature-normalised, drift-normalised, no feature encoder
(2-D data is already a sensible metric space).

In [ ]:
def compute_V(x, y_pos, y_neg=None, tau=0.2, return_weights=False):
    N = x.shape[0]
    self_neg = y_neg is None
    if self_neg: y_neg = x
    lp = -torch.cdist(x, y_pos) / tau
    ln = -torch.cdist(x, y_neg) / tau
    if self_neg: ln = ln - torch.eye(N, device=x.device) * 1e6
    logit = torch.cat([lp, ln], 1)
    A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2))
    P = y_pos.shape[0]
    A_pos, A_neg = A[:, :P], A[:, P:]
    V = (A_pos * A_neg.sum(1, keepdim=True)) @ y_pos - (A_neg * A_pos.sum(1, keepdim=True)) @ y_neg
    return (V, logit.softmax(-1)) if return_weights else V

def feature_scale(f, fp):
    with torch.no_grad():
        return (torch.cdist(f, fp).mean() / math.sqrt(f.shape[-1])).clamp_min(1e-8)

def normalised_drift(V):
    return V / V.pow(2).mean().sqrt().clamp_min(1e-8)

def train_drift(steps=4000, N=256, lr=2e-3, taus=(0.05, 0.2, 0.8), eta=0.05, seed=0, ema_decay=0.99):
    torch.manual_seed(seed)
    g = Gen().to(device)
    opt = torch.optim.Adam(g.parameters(), lr, betas=(0.9, 0.95))
    ema = {k: v.detach().clone() for k, v in g.state_dict().items()}
    C_ = 2
    for s in range(steps):
        x, y = g(torch.randn(N, 2, device=device)), ring(N)
        S = feature_scale(x, y)
        xn, yn = x / S, y / S
        with torch.no_grad():
            V = sum(normalised_drift(compute_V(xn, yn, tau=t * math.sqrt(C_))) for t in taus)
            target = xn + eta * V
        loss = F.mse_loss(xn, target)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(g.parameters(), 2.0); opt.step()
        d = min(ema_decay, (1 + s) / (10 + s))
        with torch.no_grad():
            for k, v in g.state_dict().items(): ema[k].mul_(d).add_(v.detach(), alpha=1 - d)
    ge = Gen().to(device); ge.load_state_dict(ema)
    return ge

## 2. The head-to-head

Same 4000 steps, same batch size, same MLP width. The numbers below are one seed; §4 repeats the two 1-NFE
non-diffusion methods over several.

In [ ]:
@torch.no_grad()
def gen_samples(g, n=20000):
    return g(torch.randn(n, 2, device=device))

results, samples = {}, {}
t0 = time.time(); ddpm = train_ddpm();  t_ddpm = time.time() - t0
t0 = time.time(); flow = train_flow();  t_flow = time.time() - t0
t0 = time.time(); gan  = train_gan();   t_gan  = time.time() - t0
t0 = time.time(); gmmn = train_mmd();   t_mmd  = time.time() - t0
t0 = time.time(); drft = train_drift(); t_drf  = time.time() - t0

samples["DDPM (200 NFE)"]    = sample_ddpm(ddpm, 20000, 200)
samples["DDPM (10 NFE)"]     = sample_ddpm(ddpm, 20000, 10)
samples["Flow (50 NFE)"]     = sample_flow(flow, 20000, 50)
samples["Flow (2 NFE)"]      = sample_flow(flow, 20000, 2)
samples["GAN (1 NFE)"]       = gen_samples(gan)
samples["MMD (1 NFE)"]       = gen_samples(gmmn)
samples["Drifting (1 NFE)"]  = gen_samples(drft)
samples["real data"]         = ring(20000)

print(f"{'method':22s} {'TV':>6s} {'covered':>8s} {'balance':>8s} {'on-manif':>9s}")
for k, v in samples.items():
    c, b, o = mode_stats(v)
    print(f"{k:22s} {tv2d(v, ref):6.3f} {c:8d} {b:8.3f} {o:9.3f}")
print(f"\ntrain wall-clock (s): ddpm {t_ddpm:.0f}  flow {t_flow:.0f}  gan {t_gan:.0f}  "
      f"mmd {t_mmd:.0f}  drift {t_drf:.0f}")

fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for a, (k, v) in zip(ax.ravel(), samples.items()):
    s = v[:4000].cpu()
    a.scatter(s[:, 0], s[:, 1], s=1.2, alpha=0.35, lw=0)
    a.set_title(k, fontsize=10); a.set_xlim(-1.5, 1.5); a.set_ylim(-1.5, 1.5)
    a.set_xticks([]); a.set_yticks([]); a.set_aspect("equal")
plt.tight_layout()

Two calibration points before reading anything into these numbers. The TV **noise floor** is ~0.05 (two independent
20k draws), so nothing here is converged — 4000 steps on a 3-layer MLP is a sketch, not a benchmark. And every method
got the *same* budget with *no* per-method tuning, which is itself part of the comparison.

What the table says:

* **DDPM at 200 NFE** is the accuracy leader, as it should be. At 10 NFE it degrades (TV 0.42, on-manifold 0.87):
  the discretisation error of the reverse chain is the entire story of one-step diffusion research.
* **Flow matching at 50 NFE** lands between them. At **2 NFE it fails outright** — on-manifold collapses to a few
  percent while all 8 modes are nominally "covered". The marginal velocity field at $t\approx0$ averages over every
  mode a point could go to, so a 2-step Euler solve aims at a *mean of modes* and cannot recover. This is the exact
  failure that consistency models and MeanFlow exist to fix, and it is why "just use fewer steps" is not a solution.
* **Drifting** is the best of the three 1-NFE methods on both TV and balance, at a single forward pass, and its
  on-manifold fraction matches 200-NFE DDPM.
* **MMD** is close behind. On a 2-D ring it *should* be: the space is low-dimensional, Euclidean distance is
  meaningful, and the generator is initialised near the data, so the pathology of §3 never bites. The gap widens
  exactly where those conditions fail.
* **The GAN is the weakest here**, and its failure is diffuseness (on-manifold 0.62–0.73), not collapse. §4 shows the
  other half of that story: remove the $R_1$ regulariser and it becomes the *sharpest* model in the table while
  starting to drop modes. Neither setting is wrong; picking between them is the tuning cost that the other methods
  do not charge you.

## 3. Drifting is normalised MMD

Take the Gaussian kernel $k(\mathbf{x},\mathbf{y})=\exp(-\|\mathbf{x}-\mathbf{y}\|^2/2\tau^2)$ and the (biased)
squared MMD between generated $\{\mathbf{x}_i\}_{i\le N}$ and data $\{\mathbf{y}_j\}_{j\le M}$,

$$\widehat{\mathrm{MMD}}^2 = \frac{1}{N^2}\sum_{i,i'}k(\mathbf{x}_i,\mathbf{x}_{i'}) - \frac{2}{NM}\sum_{i,j}k(\mathbf{x}_i,\mathbf{y}_j) + \text{const}.$$

Since $\nabla_{\mathbf{x}}k(\mathbf{x},\mathbf{y}) = \tau^{-2}k(\mathbf{x},\mathbf{y})(\mathbf{y}-\mathbf{x})$, the
**descent** direction is

$$-\tfrac{\tau^2}{2}\,\frac{\partial \widehat{\mathrm{MMD}}^2}{\partial \mathbf{x}_i}
= \underbrace{\frac{1}{NM}\sum_j k(\mathbf{x}_i,\mathbf{y}_j)(\mathbf{y}_j-\mathbf{x}_i)}_{\text{attraction to data}}
\;-\; \underbrace{\frac{1}{N^2}\sum_{i'} k(\mathbf{x}_i,\mathbf{x}_{i'})(\mathbf{x}_{i'}-\mathbf{x}_i)}_{\text{repulsion from own samples}}$$

which is **the drifting field with the softmax denominators deleted** (compare §3.2 of `THEORY.md`). Let us check
that claim against autograd rather than trusting the algebra.

In [ ]:
def V_unnorm(x, y_pos, y_neg=None, tau=0.3):
    """Raw kernel weights, no partition function.
    With y_neg=None (self-negatives) this equals -(tau^2/2) * d MMD^2 / dx exactly."""
    N = x.shape[0]
    y_neg = x if y_neg is None else y_neg
    M, K = y_pos.shape[0], y_neg.shape[0]
    kp = torch.exp(-torch.cdist(x, y_pos).pow(2) / (2 * tau * tau))
    kn = torch.exp(-torch.cdist(x, y_neg).pow(2) / (2 * tau * tau))
    att = (kp @ y_pos - kp.sum(1, keepdim=True) * x) / (N * M)
    rep = (kn @ y_neg - kn.sum(1, keepdim=True) * x) / (N * K)
    return att - rep

def V_norm(x, y_pos, y_neg=None, tau=0.3):
    """The same interaction, softmax-normalised along both axes: the drifting field."""
    self_neg = y_neg is None
    y_neg = x if self_neg else y_neg
    lp = -torch.cdist(x, y_pos).pow(2) / (2 * tau * tau)
    ln = -torch.cdist(x, y_neg).pow(2) / (2 * tau * tau)
    if self_neg:
        ln = ln - torch.eye(x.shape[0], device=x.device) * 1e6
    logit = torch.cat([lp, ln], 1)
    A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2))
    P = y_pos.shape[0]; Ap, An = A[:, :P], A[:, P:]
    return (Ap * An.sum(1, keepdim=True)) @ y_pos - (An * Ap.sum(1, keepdim=True)) @ y_neg

tau = 0.3
x = (0.6 * torch.randn(64, 2, device=device)).requires_grad_(True)
y = ring(64)
loss = (torch.exp(-torch.cdist(x, x).pow(2) / (2 * tau**2)).sum() / 64**2
        - 2 * torch.exp(-torch.cdist(x, y).pow(2) / (2 * tau**2)).sum() / 64**2)
g_auto = -0.5 * tau**2 * torch.autograd.grad(loss, x)[0]
g_hand = V_unnorm(x.detach(), y, tau=tau)
print("max |autograd MMD descent - hand-built unnormalised field| =",
      (g_auto - g_hand).abs().max().item())
print("relative                                                   =",
      ((g_auto - g_hand).norm() / g_auto.norm()).item())

Identical to floating-point precision. So MMD and drifting differ in exactly one place: **the denominator**. And
that denominator is what makes the difference between a usable field and an unusable one, for two separate reasons.

**(a) The vanishing gradient far from data.** The unnormalised attraction carries a factor
$k(\mathbf{x},\mathbf{y})$ that decays like $e^{-d^2/2\tau^2}$, so a sample far from every data point is *stranded*.
The normalised attraction $\sum_j\bar k(\mathbf{x},\mathbf{y}_j)(\mathbf{y}_j-\mathbf{x})$ divides by that same
decaying quantity: it is a convex combination of the positives minus $\mathbf{x}$, so its magnitude is simply the
distance to the data, however far away you are. Attraction only, in float64 so the decay is visible rather than
flushed to zero:

In [ ]:
y = ring(256).double()
print(f"{'offset':>8s} {'|att_unnorm|':>14s} {'|att_norm|':>12s} {'ratio':>11s}")
for off in [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]:
    x = ring(256).double() + torch.tensor([off, 0.0], device=device, dtype=torch.float64)
    d2 = torch.cdist(x, y).pow(2)
    k = torch.exp(-d2 / (2 * tau * tau))
    au = ((k @ y - k.sum(1, keepdim=True) * x) / y.shape[0]).norm(dim=-1).mean().item()
    an = ((-d2 / (2 * tau * tau)).softmax(-1) @ y - x).norm(dim=-1).mean().item()
    print(f"{off:8.1f} {au:14.3e} {an:12.3e} {an/max(au,1e-300):11.2e}")

The normalised column is just "distance to the data", growing linearly. The unnormalised column falls off a cliff:
by an offset of 4 the ratio is $\sim4\times10^{10}$, by 8 it is $\sim10^{82}$. **In float32 the unnormalised field is
not merely small at offset 4 — it is exactly zero**, because $e^{-88}$ underflows. Softmax subtracts the row maximum
before exponentiating, so the normalised field is computed stably at any distance.

In a real run this is not a curiosity: it is why GMMN needs a bandwidth *mixture* wide enough to reach the farthest
initialisation, and why the wide bandwidths then dominate and produce the diffuse samples seen in §2. Normalisation
removes the need for the mixture to solve a range problem — Notebook 3 uses multiple $\tau$ for *resolution*, not for
reach. It is also why drifting can be initialised anywhere (Notebook 1 §10) while an MMD generator cannot.

**(b) The equilibrium is different.** Drifting's fixed point is $\mathbf{V}=\mathbf{0}$ pointwise, at $q=p$.
MMD's fixed point is $\nabla_{\mathbf{x}}\widehat{\mathrm{MMD}}^2=0$, a *global* stationarity condition on a scalar
functional — and MMD$^2$ at a fixed bandwidth is a weak metric: it is small whenever a finite set of moments in the
RKHS agrees (§8 of `THEORY.md`). This is the same distinction as between "every particle is in force balance" and
"the total energy is at a critical point", and it is why the smearing in §2 is not a tuning failure.

One more consequence, easy to miss: the drifting field is invariant to the *number* of positives and negatives,
because both sums are divided by their own partition functions. The MMD gradient scales like $1/(NM)$. Change the
batch composition and you have silently changed the MMD learning rate.

In [ ]:
# field vectors on a grid of pure query points, with fixed data and a collapsed model
gx, gy = torch.meshgrid(torch.linspace(-1.6, 1.6, 17), torch.linspace(-1.6, 1.6, 17), indexing="ij")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], -1).to(device)
y = ring(512)
neg = ring(512) * 0.35                       # a "collapsed" model: all mass near the origin

fig, ax = plt.subplots(1, 2, figsize=(11, 5.2))
for a, (name, fn) in zip(ax, [("MMD gradient (unnormalised)", V_unnorm),
                              ("drifting field (normalised)", V_norm)]):
    V = fn(grid, y, neg, tau=0.3).cpu()
    g = grid.cpu()
    a.scatter(y[:, 0].cpu(), y[:, 1].cpu(), s=3, c="0.75", lw=0)
    a.scatter(neg[:, 0].cpu(), neg[:, 1].cpu(), s=3, c="crimson", lw=0)
    a.quiver(g[:, 0], g[:, 1], V[:, 0], V[:, 1], V.norm(dim=-1), cmap="viridis", angles="xy")
    a.set_title(f"{name}\nmean |V| = {V.norm(dim=-1).mean():.2e}, "
                f"corner |V| = {V.norm(dim=-1)[0]:.2e}", fontsize=10)
    a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

Grey = data (positives), red = a collapsed model's samples (negatives), arrows = the field a query point at that
location would feel. Both panels point the same way near the data — same interaction. But the unnormalised panel has
nothing at all in the corners, while the normalised panel keeps a usable magnitude everywhere, which is what lets a
single step size $\eta$ work across the whole space.

## 4. Drifting is a GAN with the discriminator replaced by a kernel

The structural parallel:

| | GAN | drifting |
|---|---|---|
| what pushes samples toward data | $\nabla_{\mathbf{x}} D(\mathbf{x})$, learned | $\mathbf{V}^{+}$, kernel-weighted mean of positives |
| what stops collapse | $D$ learns to recognise the collapsed mode and penalise it | $\mathbf{V}^{-}$, explicit repulsion from other samples |
| optimality condition | Nash equilibrium of a two-player game | $\mathbf{V}=\mathbf{0}$, a fixed point of one map |
| what can go wrong | $D$ wins (vanishing $G$ gradient) or $G$ wins (mode collapse); oscillation | $\tau$ badly scaled (Notebook 3 §8) |
| gradient at the optimum | nonzero, held in balance by the other player | exactly zero |

The critical difference is the *last row*. In a GAN, both players are still pushing at the optimum; stability is a
property of the dynamics, not of the objective, and it is famously fragile. In drifting, equilibrium is a genuine
zero of a fixed field: there is nothing left to oscillate. That is the whole reason a non-adversarial one-step method
is worth wanting.

Anti-collapse is also *structural* rather than *learned*. A GAN discriminator has to first notice a mode has been
dropped, which takes gradient steps and may never happen if $D$ is capacity-limited. Drifting's repulsion is there
from step 0 and cannot be defeated by the generator.

Cheapest way to see this: run both across seeds and look at the variance, not the mean.

In [ ]:
CONFIGS = [("GAN (R1=0.5)",   lambda seed: train_gan(steps=4000, lr=1e-3, r1=0.5, seed=seed)),
           ("GAN (R1=0)",     lambda seed: train_gan(steps=4000, lr=1e-3, r1=0.0, seed=seed)),
           ("MMD",            lambda seed: train_mmd(steps=4000, seed=seed)),
           ("Drifting",       lambda seed: train_drift(steps=4000, seed=seed))]
SEEDS = [0, 1, 2]

rows = []
for name, fn in CONFIGS:
    for seed in SEEDS:
        s_ = gen_samples(fn(seed), 20000)
        c, b, o = mode_stats(s_)
        rows.append((name, seed, tv2d(s_, ref), c, b, o))

print(f"{'method':14s} {'seed':>5s} {'TV':>7s} {'covered':>8s} {'balance':>8s} {'on-manif':>9s}")
for r in rows: print(f"{r[0]:14s} {r[1]:5d} {r[2]:7.3f} {r[3]:8d} {r[4]:8.3f} {r[5]:9.3f}")
print()
print(f"{'method':14s} {'TV mean+/-sd':>16s} {'min covered':>12s} {'balance mean+/-sd':>20s} {'on-manif':>9s}")
for name, _ in CONFIGS:
    v = np.array([[r[2], r[3], r[4], r[5]] for r in rows if r[0] == name])
    print(f"{name:14s} {v[:,0].mean():8.3f} +/-{v[:,0].std():5.3f} {int(v[:,1].min()):12d} "
          f"{v[:,2].mean():12.3f} +/-{v[:,2].std():5.3f} {v[:,3].mean():9.3f}")

The two GAN rows are the finding. With $R_1$ the GAN is *stable* — all 8 modes, every seed, `balance` within 0.005 —
but blurry: over a quarter of its mass sits off the data manifold. Turn $R_1$ off and on two of the three seeds it
sharpens dramatically (on-manifold 0.95 and 0.97), while `balance` degrades and its seed-to-seed spread grows
twelvefold, and one seed drops a mode entirely. You are choosing a point on a stability/sharpness curve, per problem,
by hand, and the diagnostic that tells you where you landed (`covered`, `balance`) is one you had to know to compute
in the first place.

Drifting has no equivalent knob. Its failure modes (Notebook 3 §8) are all *diagnosable from the four logged numbers*
and none of them are a race between two networks — the worst that mis-scaled $\tau$ does is make the field weak or
myopic, not unstable. Note also which column drifting wins by the largest margin: `balance` at 0.999–1.000 with
essentially zero seed-to-seed variance. That is repulsion doing its job.

MMD sits in between: non-adversarial like drifting, so also stable, but with the diffuseness that §3 predicts.

## 5. The NFE axis

Diffusion and flow buy their stability with a time variable, and the time variable is what costs $N$ evaluations.
Plot quality against NFE for a single trained model of each kind, and drifting is a horizontal line at $N=1$.

In [ ]:
nfes = [1, 2, 4, 8, 16, 32, 64, 128, 200]
tv_ddpm = [tv2d(sample_ddpm(ddpm, 8000, n), ref) for n in nfes]
tv_flow = [tv2d(sample_flow(flow, 8000, n), ref) for n in nfes]
tv_drft = tv2d(gen_samples(drft, 8000), ref)
tv_gan  = tv2d(gen_samples(gan, 8000), ref)

plt.figure(figsize=(6.2, 4))
plt.semilogx(nfes, tv_ddpm, "o-", label="DDPM")
plt.semilogx(nfes, tv_flow, "s-", label="Flow matching")
plt.scatter([1], [tv_drft], marker="*", s=220, c="crimson", zorder=5, label="Drifting")
plt.scatter([1], [tv_gan], marker="D", s=60, c="green", zorder=5, label="GAN")
plt.axhline(tv2d(ring(8000), ref), ls=":", c="0.5", label="sampling noise floor")
plt.xlabel("NFE (network evaluations per sample)"); plt.ylabel("TV to reference")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()

Read this plot as the map of the whole field. Everything to the right is bought with compute. The one-step methods
sit on the left edge, and the entire research programme of consistency models, MeanFlow, and distillation is about
dragging the diffusion curve leftward *without* losing its quality. Drifting arrives at the left edge from a
different direction: it never had a time variable to remove.

The honest caveat on this plot: a 2-D toy flatters one-step methods. What makes drifting non-obvious is that the
same result holds at ImageNet scale — 1.54 FID at 1 NFE for a 463M model, against 1.42 for a 675M SiT+REPA at
$250\times2$ NFE (`THEORY.md` §14). The mechanism that fails to scale on this toy would show up there.

## 6. Where each method actually lives

Consolidating (`THEORY.md` §14) with what we measured:

| | inference | training signal | time variable | adversarial | main failure mode | main cost |
|---|---|---|---|---|---|---|
| **DDPM / score SDE** | $N$ NFE | denoise at random $t$ | yes | no | discretisation at low NFE | inference compute |
| **Flow matching** | $N$ NFE | velocity regression | yes | no | averaged velocity at low NFE | inference compute |
| **Consistency / MeanFlow** | 1–4 NFE | self-consistency across $t$ | yes | no | delicate schedules | training complexity |
| **Distillation** | 1 NFE | match a teacher | inherited | sometimes | inherits teacher's ceiling | needs a teacher |
| **GAN** | 1 NFE | discriminator gradient | no | **yes** | collapse, oscillation | tuning fragility |
| **MMD** | 1 NFE | kernel-discrepancy gradient | no | no | weak metric, vanishing gradient | diffuse samples |
| **Drifting** | **1 NFE** | regression onto a drifted sample | **no** | no | metric/$\tau$ mis-scaling | needs a good metric space |

The column that matters is the last one. Every method pays somewhere:

* diffusion pays at **inference**, forever, per sample;
* GANs pay in **tuning risk**, unpredictably;
* distillation pays by **needing a teacher**, so it inherits a ceiling;
* drifting pays by **needing a metric space in which the kernel is meaningful** — the feature encoder of Notebook 4.

That last cost is real and is the method's biggest open liability (`THEORY.md` §15.4). It is also the most *portable*
cost: for images it is an SSL encoder; for molecules it can be pairwise distances (Notebook 8); for control
representations the paper found no encoder is needed at all. Unlike a teacher model, it does not have to be
generative.

### What drifting is not

Worth being precise, because the surface similarities invite over-claiming:

* **Not a distillation method.** No teacher, trained from scratch. The stop-gradient target is the model's *own*
  drifted samples, which is self-distillation in the sense of Notebook 0c, not teacher-student distillation.
* **Not a GAN.** Nothing is trained to oppose the generator. The "critic" is a fixed kernel, and the equilibrium is a
  zero, not a saddle.
* **Not an energy-based model.** The Laplace field is not a gradient field (`THEORY.md` §7.3), so there is no
  potential whose minimisation drifting performs — it is a genuinely non-conservative dynamics.
* **Not a moment-matching method**, though the algebra of §3 makes it look like one. The normalisation changes both
  the dynamics *and* the identifiability question (§6 vs §8 of `THEORY.md`).

## Summary

* On the same target, budget and architecture with no per-method tuning: drifting is the best of the 1-NFE methods
  and reaches 200-NFE DDPM's on-manifold fraction at **one** forward pass. Its mode balance is 0.999+ with
  essentially no seed-to-seed variance.
* **MMD's gradient is the un-normalised drifting field** — verified against autograd to machine precision. The
  softmax denominators fix the vanishing-gradient pathology (a factor $10^{15}$ at 4 units from the data) and change
  the equilibrium from a global scalar condition to a pointwise one.
* **Drifting stands to GANs as a fixed kernel stands to a learned discriminator.** The anti-collapse pressure is
  structural, present at step 0, and cannot be defeated by the generator. The measured cost of *not* having it: with
  $R_1$ the GAN is stable but blurry, without it sharp but starting to drop modes — a per-problem hand-tuned choice.
* Diffusion and flow trade inference compute for stability via a time variable. Drifting never introduces one, which
  is why its NFE curve is a point rather than a curve.

Next: [7. Latent Drifting and Learned Priors](<7. Latent Drifting and Learned Priors.ipynb>) — moving the field out
of data space, and the *moving prior* problem you hit when the space is itself being learned.